In [ ]:
!pip install pandas -q
!pip install seaborn  -q

In [1]:
import pandas as pd
import numpy as np

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score


### EDA 

In [2]:
df = pd.read_csv("data/compustat.csv")

print(df.shape)


(620141, 58)


In [8]:
df.head()

,costat,curcdq,datafmt,indfmt,consol,tic,datadate,gvkey,conm,conml,...,dltisy,dvy,oancfy,oibdpy,scstkcy,mkvaltq,prchq,prclq,year,quarter
705,A,USD,STD,INDL,C,AIR,2010-02-28,1004,AAR CORP,AAR Corp,...,0.000,0.0,94.647,NaN,NaN,885.0870,26.08,18.77,2010,1
9876,A,USD,STD,INDL,C,AIR,2010-05-31,1004,AAR CORP,AAR Corp,...,63.545,0.0,153.156,NaN,NaN,777.8348,25.91,18.03,2010,2
19004,A,USD,STD,INDL,C,AIR,2010-08-31,1004,AAR CORP,AAR Corp,...,0.000,0.0,7.218,NaN,NaN,609.2237,19.46,14.91,2010,3
28067,A,USD,STD,INDL,C,AIR,2010-11-30,1004,AAR CORP,AAR Corp,...,0.000,0.0,29.812,NaN,NaN,974.4917,24.60,15.56,2010,4
36979,A,USD,STD,INDL,C,AIR,2011-02-28,1004,AAR CORP,AAR Corp,...,0.000,0.0,59.902,NaN,NaN,1084.5592,29.05,24.68,2011,1


In [3]:
print(df.duplicated().sum(), "duplicate rows found and deleted")
df = df.drop_duplicates()

0 duplicate rows found and deleted


In [4]:
df['datadate'] = pd.to_datetime(df['datadate'])

df = df.sort_values(['gvkey', 'datadate'])

# year-quarter index
df['year'] = df['datadate'].dt.year
df['quarter'] = df['datadate'].dt.quarter


In [5]:
obs_per_firm = df.groupby('gvkey')['datadate'].count()

# Keep firms with at least 12 quarters (3 years)
valid_firms = obs_per_firm[obs_per_firm >= 12].index
df = df[df['gvkey'].isin(valid_firms)]


In [6]:
targets = ['oancfy', 'cheq']

df[targets].isna().mean()

oancfy    0.337925
cheq      0.316631
dtype: float64

In [7]:
missing = df.isna().mean().sort_values(ascending=False)

missing.head(15)

findltq    0.997650
udoltq     0.988637
scstkcy    0.973903
deraltq    0.890160
derlltq    0.868841
cshopq     0.684350
xrdq       0.680577
xaccq      0.665935
ipodate    0.584330
dd1q       0.529449
npq        0.475765
lltq       0.473970
drcq       0.472951
ivltq      0.460721
wcapq      0.450017
dtype: float64

### preprocessing to be done
- normalize
- imputation
- check stationary characteristics
- test feature selection methods

### BASELINE

- predict quaters (liquidity )
- predict early (cash flow)

In [8]:
lags = 4  #use last 4 quarters
features = []

for target in targets:
    for l in range(1, lags + 1):
        col_name = f"{target}_lag{l}"
        df[col_name] = df.groupby('gvkey')[target].shift(l)
        features.append(col_name)

df_model = df.dropna(subset=features + targets).copy()

In [9]:
split_date = df_model['datadate'].quantile(0.8)

train = df_model[df_model['datadate'] <= split_date]
test = df_model[df_model['datadate'] > split_date]

X_train = train[features]
X_test = test[features]

In [ ]:
results = {}

for target in targets:
    y_train = train[target]
    y_test = test[target]
    
    rf = RandomForestRegressor(
        n_estimators=200,
        max_depth=None,
        random_state=42,
        n_jobs=-1
    )
    
    rf.fit(X_train, y_train)
    preds = rf.predict(X_test)
    
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    r2 = r2_score(y_test, preds)
    results[target] = {
        "RMSE": rmse,
        "R2": r2
    }
